# 06 — Measuring age and metallicity: fitting one SSP with FaSTAR

*Lecture 1: stellar populations*

We make a **mock galaxy spectrum** with known age and metallicity, add the effects from notebook 05 (velocity dispersion, noise), and try to get the parameters back by comparing it with FaSTAR models:

$$\chi^2(t, Z) = \sum_\mathrm{pixels} \left(\frac{F_\mathrm{obs} - A\,F_\mathrm{model}(t, Z)}{\sigma_\mathrm{noise}}\right)^2 .$$

The amplitude $A$ (the galaxy's mass and distance) is solved for exactly at each step (it is linear). You will meet the **age–metallicity degeneracy** of the lecture in numbers.

Tools: FaSTAR, numpy, scipy (`scipy.optimize.minimize`). A model takes ~0.2 s, so the grid below needs ~1–2 minutes.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Azeret/stellar-populations-lectures/blob/main/notebooks/06_fitting_an_SSP_with_FaSTAR.ipynb)  ← run this notebook in your browser, nothing to install

In [ ]:
# Running in Google Colab? This cell fetches the course code and installs fastar-astro.
# On your own computer it does nothing.
import sys, os, subprocess
if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/stellar-populations-lectures'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Azeret/stellar-populations-lectures.git',
                        '/content/stellar-populations-lectures'], check=True)
    os.chdir('/content/stellar-populations-lectures/notebooks')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastar-astro'], check=True)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # makes the small 'stellarpops' package importable
import numpy as np
import matplotlib.pyplot as plt
from stellarpops.style import use_slide_style
use_slide_style(dark=False, scale=0.7)       # dark=True gives the lecture-slide look

from scipy.optimize import minimize
from fastar import IntegratedSSPSynthesizer, imf
from stellarpops.spectra import log_rebin, broaden_velocity, add_noise, normalise

ssp = IntegratedSSPSynthesizer(imf_function=imf.kroupa)
wave = np.array(ssp.wave)
ln_lam, _, velscale = log_rebin(wave, np.ones_like(wave))
lam = np.exp(ln_lam)
FIT = (lam > 3800) & (lam < 7200)          # pixels used in the fit

def model(age, met, sigma):
    # FaSTAR SSP, log-rebinned and broadened to the galaxy's velocity dispersion
    s = np.array(ssp.synthesize(age=float(age), met=float(met)))
    s = np.interp(ln_lam, np.log(wave), s)
    return broaden_velocity(s / np.median(s), velscale, sigma)

## 1. A mock galaxy

In [ ]:
TRUE = dict(age=5.0, met=-0.3)
SIGMA = 150.0          # km/s, assumed known (pPXF measures it, notebook 07)
SNR = 50

clean = model(TRUE['age'], TRUE['met'], SIGMA)
obs, err = add_noise(clean, SNR, rng=42)

plt.figure(figsize=(11, 4))
plt.plot(lam[FIT], obs[FIT], lw=0.6, label=f'mock: {TRUE["age"]} Gyr, [M/H] = {TRUE["met"]}, S/N = {SNR}')
plt.xlabel(r'wavelength [$\AA$]'); plt.legend()

## 2. χ² with the best amplitude

For fixed $(t, Z)$ the best amplitude is $A = \sum (F_\mathrm{obs} F_\mathrm{mod}/\sigma^2) / \sum (F_\mathrm{mod}^2/\sigma^2)$ — a one-line linear fit.

In [ ]:
def chi2(age, met, data=obs, noise=err, sigma=SIGMA, mask=FIT):
    m = model(age, met, sigma)[mask]
    d, e = data[mask], noise[mask]
    A = np.sum(d * m / e**2) / np.sum(m**2 / e**2)
    return np.sum(((d - A * m) / e) ** 2)

print('chi2 at the truth:', round(chi2(**TRUE)), 'for', FIT.sum(), 'pixels')
print('chi2 at 10 Gyr, solar:', round(chi2(10.0, 0.0)))

## 3. A grid of models and the posterior

We evaluate χ² on a grid and turn it into a probability, $P \propto e^{-\chi^2/2}$ (flat priors).

In [ ]:
ages = np.geomspace(1.0, 13.5, 22)
mets = np.linspace(-1.0, 0.3, 14)
chi = np.array([[chi2(a, m) for m in mets] for a in ages])
post = np.exp(-0.5 * (chi - chi.min()))

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.contourf(mets, np.log10(ages), post, levels=[0.01, 0.135, 0.6, 1.0], cmap='Blues')
ax.contour(mets, np.log10(ages), chi - chi.min(), levels=[2.3, 6.2, 11.8], colors='k', linewidths=1)
ax.plot(TRUE['met'], np.log10(TRUE['age']), 'r*', ms=18, label='truth')
i, j = np.unravel_index(chi.argmin(), chi.shape)
ax.plot(mets[j], np.log10(ages[i]), 'kx', ms=12, mew=3, label='best grid point')
ax.set_xlabel('[M/H]'); ax.set_ylabel('log age [Gyr]'); ax.legend()

**Question.** Which way does the probability contour lean? Relate it to the "older or more metal-rich?" slide.

## 4. A proper minimisation

The grid is coarse. Starting from the best grid point, we let `scipy.optimize.minimize` (Nelder–Mead, no derivatives needed) find the minimum in $(\log t, [M/H])$.

In [ ]:
def f(p):
    la, met = p
    if not (np.log10(0.05) < la < np.log10(13.9) and -2.4 < met < 0.29):
        return 1e30
    return chi2(10 ** la, met)

res = minimize(f, x0=[np.log10(ages[i]), mets[j]], method='Nelder-Mead',
               options=dict(xatol=1e-3, fatol=0.5))
print(f'best fit: age = {10 ** res.x[0]:.2f} Gyr, [M/H] = {res.x[1]:+.3f}   (truth {TRUE["age"]} Gyr, {TRUE["met"]})')
print('model evaluations:', res.nfev)

## 5. How good is the answer? Repeat with new noise

Fit 6 noise realisations at three S/N values (this takes a few minutes). The scatter of the results is the real uncertainty — and it lies along the degeneracy.

In [ ]:
results = {}
for snr in [15, 50, 150]:
    out = []
    for seed in range(6):
        d, e = add_noise(clean, snr, rng=100 + seed)
        g = lambda p: 1e30 if not (-1.3 < p[0] < 1.14 and -2.4 < p[1] < 0.29) else chi2(10 ** p[0], p[1], data=d, noise=e)
        out.append(minimize(g, x0=[np.log10(TRUE['age']), TRUE['met']], method='Nelder-Mead',
                            options=dict(xatol=2e-3, fatol=0.5)).x)
    results[snr] = np.array(out)

fig, ax = plt.subplots(figsize=(7, 5.5))
for snr, r in results.items():
    ax.plot(r[:, 1], r[:, 0], 'o', ms=9, label=f'S/N = {snr}')
ax.plot(TRUE['met'], np.log10(TRUE['age']), 'k*', ms=18, label='truth')
ax.set_xlabel('[M/H]'); ax.set_ylabel('log age [Gyr]'); ax.legend()

## 6. Colour alone, or lines?

Smooth away all lines (a 300 Å running mean keeps only the continuum shape, i.e. the "colours") and repeat the grid. What happens to the degeneracy?

In [ ]:
def continuum(s, width_pix=300):
    k = np.ones(width_pix) / width_pix
    return np.convolve(np.concatenate([np.full(width_pix, s[0]), s, np.full(width_pix, s[-1])]), k, 'same')[width_pix:-width_pix]

obs_c = continuum(obs)
def chi2_cont(age, met):
    m = continuum(model(age, met, SIGMA))[FIT]
    d, e = obs_c[FIT], err[FIT] / 10      # smoothing beats down the noise: use a smaller error
    A = np.sum(d * m / e**2) / np.sum(m**2 / e**2)
    return np.sum(((d - A * m) / e) ** 2)

chi_c = np.array([[chi2_cont(a, m) for m in mets] for a in ages])
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.contour(mets, np.log10(ages), chi_c - chi_c.min(), levels=[2.3, 6.2, 11.8], colors='C1')
ax.contour(mets, np.log10(ages), chi - chi.min(), levels=[2.3, 6.2, 11.8], colors='k')
ax.plot(TRUE['met'], np.log10(TRUE['age']), 'r*', ms=18)
ax.set_xlabel('[M/H]'); ax.set_ylabel('log age [Gyr]'); ax.set_title('black: full spectrum, orange: continuum only')

## Exercises

1. Fit with the wrong velocity dispersion (e.g. `sigma=100` in `chi2`). Which parameter is biased, and why?
2. Make the mock with a bottom-heavy IMF: `IntegratedSSPSynthesizer(imf_function=imf.single_power_law)` with `imf_params={'alpha': 3.0}`, and fit it with Kroupa models. Is the fit bad? Are age and metallicity biased?
3. Mask everything except Hβ, Mg b and Fe5270/5335 (`mask=` argument of `chi2`). Do you lose much precision?
4. Real galaxies are not one SSP — that is notebook 07.